# HateGuard - Deploy on Google Colab

This notebook acts as an automated setup to run the HateGuard product.
It starts both the FastAPI backend and the React UI, and gives you a fully public Cloudflare Tunnel link to use the site!

In [ ]:
import os
# 1. Clone the repository
if not os.path.exists('hate-comment-dectection'):
    !git clone https://github.com/ATANU28-bit/hate-comment-dectection.git
%cd hate-comment-dectection

In [ ]:
# 2. Install dependencies & tools
!sudo apt update && sudo apt install ffmpeg -y
!pip install -r requirements.txt

# Install UI dependencies
%cd ui
!npm install
%cd ..

# Download Cloudflare Tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
import subprocess
import time
import threading
import queue

# Helper to retrieve the Cloudflare URL from stderr output
def get_cloudflare_url(proc):
    q = queue.Queue()
    def enqueue_output(out, queue):
        for line in iter(out.readline, b''):
            line_str = line.decode('utf-8')
            if "trycloudflare.com" in line_str:
                # Extract URL
                url_start = line_str.find("https://")
                url_end = line_str.find(" ", url_start)
                if url_end == -1: url_end = len(line_str)
                url = line_str[url_start:url_end].strip()
                queue.put(url)
        out.close()
        
    t = threading.Thread(target=enqueue_output, args=(proc.stderr, q))
    t.daemon = True
    t.start()
    try:
        return q.get(timeout=20)
    except queue.Empty:
        return None

print("Starting Backend...")
backend = subprocess.Popen(["uvicorn", "src.api:app", "--host", "127.0.0.1", "--port", "8000"])
cf_backend = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stderr=subprocess.PIPE)

backend_url = get_cloudflare_url(cf_backend)
if not backend_url:
    print("Failed to create tunnel for backend!")
else:
    print(f"⭐ BACKEND API URl: {backend_url}")

    # Set Backend URL for Frontend dynamic access before starting Vite
    with open("ui/.env", "w", encoding="utf-8") as f:
        f.write(f"VITE_API_URL={backend_url}\n")
    
    print("Starting Frontend...")
    frontend = subprocess.Popen(["npm", "run", "dev", "--prefix", "ui", "--", "--host", "127.0.0.1", "--port", "5173"])
    cf_frontend = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:5173"], stderr=subprocess.PIPE)
    
    frontend_url = get_cloudflare_url(cf_frontend)
    
    if not frontend_url:
        print("Failed to create tunnel for frontend!")
    else:
        print("\n=========================================================")
        print("🚀 YOUR APPLICATION IS READY!")
        print(f"👉 OPEN THIS LINK FOR THE UI: {frontend_url}")
        print("=========================================================\n")

    try:
        frontend.wait() # Keep cells running
    except KeyboardInterrupt:
        print("Shutting down servers...")
        backend.terminate()
        frontend.terminate()
        cf_backend.terminate()
        cf_frontend.terminate()